In [ ]:
import os
import sys

# to setup import paths add project root dirs to sys.path
sys.path.append(os.path.join(os.getcwd(), "..", ".."))
from baseVR.base_functionality import init_import_paths
init_import_paths()


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# %matplotlib qt
# %matplotlib widget

from analytics_processing import analytics
import analytics_processing.analytics_constants as C
from CustomLogger import CustomLogger as Logger

from dashsrc.plot_components.plot_wrappers.data_selection import group_filter_data

from analytics_processing.modality_loading import session_modality_from_nas
from analytics_processing.sessions_from_nas_parsing import sessionlist_fullfnames_from_args
from analytics_processing.sessions_from_nas_parsing import fullfnames2snames


In [ ]:
animal_ids = [6]
paradigm_ids = [1100]
session_ids = None

output_dir = "./outputs/experimental/"
nas_dir = C.device_paths()[0]
Logger().init_logger(None, None, logging_level="INFO")

In [ ]:
session_dirs = sessionlist_fullfnames_from_args(paradigm_ids, animal_ids, session_ids)[0]
session_dirs

In [ ]:

from matplotlib.animation import FuncAnimation
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import numpy as np
import math


def animate_body_points(df, frames=None, start_idx=0, end_idx=240,
                        interval=1000/24, embed_limit_mb=500, dpi=100,
                        segment_names=None):
    """
    Pose kinematics animation. Adapts to whatever columns are present in df,
    as produced by pose_kinematics.add_kinematic_features() with segment_names.

    Parameters
    ----------
    segment_names : dict {(p1, p2): name}, optional
        Same dict passed to add_kinematic_features. Used to map segment names
        back to keypoint pairs for drawing skeleton + arcs on the video.

    Layout
    ------
    Row 0  Segment confidence  (one line per <seg>_likelihood)
    Row 1  Segment angles      (one line per <seg>_angle, absolute heading)
    Row 2  Joint bends         (one trace per <x>_<y>_bend, stacked)
    Row 3  Movement energy + spine curvature  |  body elongation
    Row 4  Video frame with skeleton + angle arc overlays
    """
    import matplotlib as mpl
    mpl.rcParams["animation.embed_limit"] = embed_limit_mb

    df_s = df.iloc[start_idx:end_idx].reset_index(drop=True)
    N    = len(df_s)
    fi_  = np.arange(N)

    BG, PANEL, GRID = "#0d1117", "#161b22", "#21262d"
    MUTED, WHITE    = "#8b949e", "#e6edf3"
    LK_THRESH       = 0.6

    KP_COL = {"nose":"#ff6b6b","neck":"#ffd93d","right_front_paw":"#6bcb77",
              "mid_spine":"#4d96ff","late_spine":"#c77dff","right_hind_paw":"#ff9f1c"}
    PAL = ["#ff6b6b","#ffd93d","#6bcb77","#4d96ff","#c77dff","#ff9f1c",
           "#00d4ff","#ff6fc8","#aaff00","#e040fb","#40c4ff","#ff4444"]

    def get(col):
        return df_s[col].astype(float).values if col in df_s.columns \
               else np.full(N, np.nan)

    def _style(ax, title, ylabel, ylim=None):
        ax.set_facecolor(PANEL); ax.spines[:].set_color(GRID)
        ax.tick_params(colors=MUTED, labelsize=8)
        ax.xaxis.label.set_color(MUTED); ax.yaxis.label.set_color(MUTED)
        ax.grid(color=GRID, lw=0.5, alpha=0.9)
        ax.set_xlim(0, N-1); ax.set_xlabel("frame", fontsize=8, color=MUTED)
        ax.set_title(title, fontsize=9, color=WHITE, loc="left", pad=4)
        ax.set_ylabel(ylabel, fontsize=8, color=MUTED)
        if ylim: ax.set_ylim(*ylim)

    def _leg(ax, **kw):
        ax.legend(fontsize=7, facecolor=PANEL, edgecolor=GRID,
                  labelcolor=WHITE, **kw)

    # ── auto-discover columns from df ────────────────────────────────────────
    # Segment angle cols: <seg>_angle where <seg>_likelihood also exists
    seg_angle_cols = sorted([
        c for c in df_s.columns
        if c.endswith("_angle")
        and not c.endswith(("_vel","_body_angle"))
        and c.replace("_angle","_likelihood") in df_s.columns
    ])
    seg_lk_cols  = [c.replace("_angle","_likelihood") for c in seg_angle_cols]
    seg_labels   = [c.replace("_angle","").replace("_"," ") for c in seg_angle_cols]

    # Bend cols: <x>_<y>_bend
    bend_cols   = sorted([c for c in df_s.columns
                          if c.endswith("_bend")
                          and not c.endswith(("_vel","_likelihood"))])
    bend_labels = [c.replace("_bend","").replace("_"," ") for c in bend_cols]
    bend_data   = [get(c) for c in bend_cols]

    # Segment colours — consistent across rows and arcs
    seg_colors  = [PAL[i % len(PAL)] for i in range(len(seg_angle_cols))]
    bend_colors = [PAL[(len(seg_angle_cols)+i) % len(PAL)]
                   for i in range(len(bend_cols))]

    # ── keypoints ─────────────────────────────────────────────────────────────
    KEYPOINTS = [k for k in KP_COL if f"{k}_x" in df_s.columns]
    lk = {kp: get(f"{kp}_likelihood") for kp in KEYPOINTS}
    xr = {kp: get(f"{kp}_x")          for kp in KEYPOINTS}
    yr = {kp: get(f"{kp}_y")          for kp in KEYPOINTS}
    xd = {kp: np.where(lk[kp] >= LK_THRESH, xr[kp], np.nan) for kp in KEYPOINTS}
    yd = {kp: np.where(lk[kp] >= LK_THRESH, yr[kp], np.nan) for kp in KEYPOINTS}

    # ── skeleton & bend→keypoint mapping from segment_names ──────────────────
    if segment_names:
        inv = {v: tuple(k) if not isinstance(k, tuple) else k
               for k, v in segment_names.items()}

        SKELETON = [inv[c.replace("_angle","")]
                    for c in seg_angle_cols
                    if c.replace("_angle","") in inv]

        def _bend_kps(col):
            """Resolve <sega>_<segb>_bend → (arm1, vtx, arm2) keypoints."""
            stem  = col[:-len("_bend")]
            parts = stem.split("_")
            for split in range(1, len(parts)):
                s1 = "_".join(parts[:split])
                s2 = "_".join(parts[split:])
                if s1 in inv and s2 in inv:
                    p1a, vtx  = inv[s1]
                    vtxb, p2b = inv[s2]
                    if vtx == vtxb:
                        return p1a, vtx, p2b
            return None
        bend_kps = [_bend_kps(c) for c in bend_cols]
    else:
        SKELETON = [("nose","neck"),("neck","right_front_paw"),
                    ("neck","mid_spine"),("mid_spine","late_spine"),
                    ("late_spine","right_hind_paw")]
        bend_kps = [None] * len(bend_cols)

    # ── scalar composites ─────────────────────────────────────────────────────
    energy     = get("movement_energy_smooth5")
    spine_curv = get("spine_curvature")
    body_elong = get("body_elongation")

    # ── figure layout ─────────────────────────────────────────────────────────
    n_bends = max(len(bend_cols), 1)
    fig = plt.figure(figsize=(16, 24), facecolor=BG, dpi=dpi)
    gs  = gridspec.GridSpec(5, 1, figure=fig,
                            height_ratios=[1.0, 1.0, 0.45*n_bends, 0.9, 5.0],
                            hspace=0.55)
    ax_lk   = fig.add_subplot(gs[0])
    ax_ang  = fig.add_subplot(gs[1])
    ax_bend = fig.add_subplot(gs[2])
    ax_comp = fig.add_subplot(gs[3])
    ax_vid  = fig.add_subplot(gs[4])

    _style(ax_lk,   "Segment Confidence", "likelihood", (0, 1.05))
    _style(ax_ang,  "Segment Angles  (0°=right, 90°=down)", "deg", (0, 360))
    _style(ax_bend, "Joint Bends  (stacked, each 0–180°)", "deg")
    _style(ax_comp, "Movement Energy  |  Spine Curvature  |  Body Elongation", "")
    ax_vid.set_facecolor("black")
    ax_vid.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    for sp in ax_vid.spines.values(): sp.set_color(GRID)

    # row 0: confidence
    ax_lk.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
    for col, lbl, c in zip(seg_lk_cols, seg_labels, seg_colors):
        ax_lk.plot(fi_, get(col), color=c, lw=1.1, label=lbl)
    ax_lk.axhline(LK_THRESH, color=WHITE, lw=0.8, ls="--", alpha=0.4,
                  label=f"≥{LK_THRESH}")
    _leg(ax_lk, ncol=min(len(seg_lk_cols)+1, 7), loc="lower right")

    # row 1: segment angles
    ax_ang.set_yticks([0, 90, 180, 270, 360])
    for col, lbl, c in zip(seg_angle_cols, seg_labels, seg_colors):
        ax_ang.plot(fi_, get(col), color=c, lw=1.0, label=lbl)
    _leg(ax_ang, ncol=min(len(seg_angle_cols), 6), loc="upper right")

    # row 2: bends — stacked with 220° offset each
    OFFSET = 220
    ytv, ytl = [], []
    for i, (col, lbl, c) in enumerate(zip(bend_cols, bend_labels, bend_colors)):
        base = i * OFFSET
        ax_bend.plot(fi_, bend_data[i] + base, color=c, lw=1.0)
        ax_bend.axhline(base,     color=c, lw=0.3, ls=":", alpha=0.3)
        ax_bend.axhline(base+180, color=c, lw=0.3, ls=":", alpha=0.3)
        ax_bend.text(N-1, base+90, f"  {lbl}", fontsize=7,
                     color=c, va="center", ha="left")
        ytv += [base, base+90, base+180]
        ytl += ["0°", "90°", "180°"]
    if bend_cols:
        ax_bend.set_ylim(-10, len(bend_cols)*OFFSET - 10)
        ax_bend.set_yticks(ytv); ax_bend.set_yticklabels(ytl, fontsize=6)

    # row 3: composites
    ax_comp.plot(fi_, energy,     color="#4ec9b0", lw=1.2, label="movement energy")
    ax_comp.plot(fi_, spine_curv, color="#ff9f1c", lw=1.0, label="spine curvature")
    ax_comp.set_ylim(bottom=0)
    ax_c2 = ax_comp.twinx()
    ax_c2.set_facecolor(PANEL); ax_c2.spines[:].set_color(GRID)
    ax_c2.tick_params(colors=MUTED, labelsize=8)
    ax_c2.plot(fi_, body_elong, color="#9cdcfe", lw=1.0, label="body elongation")
    ax_c2.axhline(1.0, color=WHITE, lw=0.6, ls="--", alpha=0.3)
    ax_c2.set_ylim(0, 1.4); ax_c2.set_ylabel("elongation", fontsize=8, color=MUTED)
    h1,l1 = ax_comp.get_legend_handles_labels()
    h2,l2 = ax_c2.get_legend_handles_labels()
    ax_c2.legend(h1+h2, l1+l2, fontsize=7, loc="upper right",
                 facecolor=PANEL, edgecolor=GRID, labelcolor=WHITE)

    # playheads
    playheads = [ax.axvline(0, color=WHITE, lw=0.9, alpha=0.45, zorder=10)
                 for ax in [ax_lk, ax_ang, ax_bend, ax_comp]]

    # video canvas
    if frames is not None:
        h_px, w_px = frames.shape[1:3]
        im_obj = ax_vid.imshow(frames[start_idx], aspect="auto",
                               extent=[0, w_px, h_px, 0])
        ax_vid.set_xlim(0, w_px); ax_vid.set_ylim(h_px, 0)
    else:
        all_x = np.concatenate([xr[kp] for kp in KEYPOINTS])
        all_y = np.concatenate([yr[kp] for kp in KEYPOINTS])
        pad = 40
        ax_vid.set_xlim(np.nanmin(all_x)-pad, np.nanmax(all_x)+pad)
        ax_vid.set_ylim(np.nanmax(all_y)+pad, np.nanmin(all_y)-pad)
        im_obj = None

    # skeleton lines
    seg_lines = [ax_vid.plot([], [], color=WHITE, lw=2.5,
                             solid_capstyle="round")[0]
                 for _ in SKELETON]

    # keypoint dots + name labels
    kp_dots = {kp: ax_vid.plot([], [], "o", color=KP_COL.get(kp,"#aaa"),
                               ms=9, markeredgecolor=WHITE, markeredgewidth=0.7,
                               zorder=6)[0]
               for kp in KEYPOINTS}
    kp_lbls = {kp: ax_vid.text(0, 0, kp.replace("_"," "),
                                fontsize=7, color=KP_COL.get(kp,"#aaa"),
                                ha="left", va="bottom", alpha=0, zorder=7)
               for kp in KEYPOINTS}

    arc_pool = []

    hud = ax_vid.text(0.01, 0.02, "", transform=ax_vid.transAxes,
                      va="bottom", ha="left", fontsize=8, color=WHITE,
                      family="monospace",
                      bbox=dict(boxstyle="round,pad=0.4", facecolor="black", alpha=0.6))

    def _arc(xv, yv, x1, y1, x2, y2, deg, label, color, r=22):
        a1 = math.degrees(math.atan2(y1-yv, x1-xv))
        a2 = math.degrees(math.atan2(y2-yv, x2-xv))
        if abs(a2-a1) > 180:
            if a2>a1: a1+=360
            else:     a2+=360
        t1,t2 = min(a1,a2), max(a1,a2)
        patch = mpatches.Arc((xv,yv), 2*r, 2*r, angle=0,
                             theta1=t1, theta2=t2, color=color, lw=2.0)
        ax_vid.add_patch(patch)
        mt = math.radians((t1+t2)/2)
        lx = xv+(r+14)*math.cos(mt); ly = yv+(r+14)*math.sin(mt)
        txt = ax_vid.text(lx, ly, f"{label}\n{deg:.0f}\u00b0",
                          fontsize=7, color=color, ha="center", va="center",
                          fontweight="bold", linespacing=1.15,
                          bbox=dict(boxstyle="round,pad=0.15",
                                    facecolor=BG, alpha=0.6, linewidth=0))
        return patch, txt

    def init():
        if im_obj is not None: im_obj.set_data(frames[start_idx])
        for ln in seg_lines: ln.set_data([], [])
        for dot in kp_dots.values(): dot.set_data([], [])
        for lbl in kp_lbls.values(): lbl.set_alpha(0)
        hud.set_text("")
        for ph in playheads: ph.set_xdata([0])

    def update(fi):
        for a in arc_pool:
            try: a.remove()
            except Exception: pass
        arc_pool.clear()

        if im_obj is not None:
            im_obj.set_data(frames[start_idx+fi])

        # skeleton
        for line, (p1,p2) in zip(seg_lines, SKELETON):
            if (p1 in xd and p2 in xd
                    and not np.isnan(xd[p1][fi])
                    and not np.isnan(xd[p2][fi])):
                line.set_data([xd[p1][fi],xd[p2][fi]], [yd[p1][fi],yd[p2][fi]])
            else:
                line.set_data([],[])

        # dots + labels
        for kp in KEYPOINTS:
            if np.isnan(xd[kp][fi]):
                kp_dots[kp].set_data([],[])
                kp_lbls[kp].set_alpha(0)
            else:
                kp_dots[kp].set_data([xd[kp][fi]], [yd[kp][fi]])
                kp_dots[kp].set_alpha(min(1.0, lk[kp][fi]))
                kp_lbls[kp].set_position((xd[kp][fi]+5, yd[kp][fi]-5))
                kp_lbls[kp].set_alpha(min(1.0, lk[kp][fi]))

        # bend arcs
        for i, (col, lbl, c) in enumerate(zip(bend_cols, bend_labels, bend_colors)):
            val = bend_data[i][fi]
            kps = bend_kps[i]
            if np.isnan(val) or kps is None:
                continue
            arm1, vtx, arm2 = kps
            if any(k not in xd or np.isnan(xd[k][fi])
                   for k in [arm1, vtx, arm2]):
                continue
            arc_pool.extend(_arc(xd[vtx][fi], yd[vtx][fi],
                                 xd[arm1][fi], yd[arm1][fi],
                                 xd[arm2][fi], yd[arm2][fi],
                                 val, lbl, c))

        hud.set_text(f"frame  {start_idx+fi:5d}\n"
                     f"energy {energy[fi]:6.1f}\n"
                     f"curv.  {spine_curv[fi]:5.1f}\u00b0\n"
                     f"elong. {body_elong[fi]:5.2f}")
        for ph in playheads: ph.set_xdata([fi])

    ani = FuncAnimation(fig, update, frames=N, init_func=init,
                        interval=interval, blit=False)
    fig.suptitle("Pose Kinematics", fontsize=13, color=WHITE,
                 fontweight="bold", y=0.999)
    plt.close(fig)
    return ani

In [ ]:
pose = analytics.get_analytics('BehaviorPose', session_names=['2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min'])
pose

In [ ]:
# ...existing code...
import math
import matplotlib.pyplot as plt
import matplotlib as mpl
plt.style.use("seaborn-v0_8-darkgrid")

cols = pose.columns.tolist()
# find keypoints by looking for *_x columns
kp_bases = sorted({c[:-2] for c in cols if c.endswith("_x")})
n_kp = len(kp_bases)

ncols = 8
nplots = len(cols) + n_kp
nrows = math.ceil(nplots / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(4.0 * ncols, 2.8 * nrows), constrained_layout=True)
axes = axes.flatten()

# hist plots for every dataframe column
for i, col in enumerate(cols):
    ax = axes[i]
    data = pose[col].dropna()
    na_frac = pose[col].isna().mean()
    if data.size == 0:
        ax.text(0.5, 0.5, "no data", ha="center", va="center", color="gray", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(col, fontsize=9, loc="left")
    else:
        if col.endswith("_likelihood"):
            ax.hist(data, bins=50, range=(0, 1), color="#4c72b0", edgecolor="white", alpha=0.95)
        else:
            ax.hist(data, bins=50, color="#4c72b0", edgecolor="white", alpha=0.95)
        ax.grid(True, ls="--", alpha=0.4)
        ax.set_xlabel(col, fontsize=8)
        ax.set_ylabel("Count", fontsize=8)
        ax.tick_params(axis="both", labelsize=7)
        ax.set_title(col, fontsize=9, loc="left")
    # small NA annotation in corner
    ax.text(0.98, 0.92, f"NA {na_frac:.1%}", transform=ax.transAxes,
            ha="right", va="top", fontsize=7, color="gray",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.6, linewidth=0))# ...existing code...
# ...existing code...
# scatter plots for each keypoint (x vs y, color by likelihood)
cmap = plt.get_cmap("RdYlGn")
for k_i, kp in enumerate(kp_bases):
    ax_idx = len(cols) + k_i
    ax = axes[ax_idx]
    x_col = f"{kp}_x"
    y_col = f"{kp}_y"
    lk_col = f"{kp}_likelihood"
    # set standard image axes for all keypoint plots
    ax.set_xlim(0, 640)
    ax.set_ylim(480, 0)  # flip y-axis to match image coordinates

    if x_col not in pose.columns or y_col not in pose.columns:
        ax.text(0.5, 0.5, "no coords", ha="center", va="center", color="gray", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(kp, fontsize=9, loc="left")
        continue
    x = pose[x_col]
    y = pose[y_col]
    valid = x.notna() & y.notna()
    na_frac = (~valid).sum() / len(pose)
    if valid.sum() == 0:
        ax.text(0.5, 0.5, "no valid points", ha="center", va="center", color="gray", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
    else:
        if lk_col in pose.columns:
            lk = pose[lk_col].fillna(0.0)
            sc = ax.scatter(x[valid], y[valid], c=lk[valid], cmap=cmap, vmin=0, vmax=1,
                            s=6, alpha=0.7, linewidths=0)
            # small colorbar to the right
            cb = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.02)
            cb.set_label("likelihood", fontsize=7)
            cb.ax.tick_params(labelsize=7)
        else:
            ax.scatter(x[valid], y[valid], color="#4c72b0", s=6, alpha=0.7)
        ax.set_xlabel("x", fontsize=8)
        ax.set_ylabel("y", fontsize=8)
        ax.set_title(kp.replace("_", " "), fontsize=9, loc="left")
        ax.grid(True, ls="--", alpha=0.4)
    # annotate NA fraction
    ax.text(0.98, 0.92, f"NA {na_frac:.1%}", transform=ax.transAxes,
            ha="right", va="top", fontsize=7, color="gray",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.6, linewidth=0))
# ...existing code...

# scatter plots for each keypoint (x vs y, color by likelihood)
cmap = plt.get_cmap("RdYlGn")
for k_i, kp in enumerate(kp_bases):
    ax_idx = len(cols) + k_i
    ax = axes[ax_idx]
    x_col = f"{kp}_x"
    y_col = f"{kp}_y"
    lk_col = f"{kp}_likelihood"
    if x_col not in pose.columns or y_col not in pose.columns:
        ax.text(0.5, 0.5, "no coords", ha="center", va="center", color="gray", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(kp, fontsize=9, loc="left")
        continue
    x = pose[x_col]
    y = pose[y_col]
    valid = x.notna() & y.notna()
    na_frac = (~valid).sum() / len(pose)
    if valid.sum() == 0:
        ax.text(0.5, 0.5, "no valid points", ha="center", va="center", color="gray", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
    else:
        if lk_col in pose.columns:
            lk = pose[lk_col].fillna(0.0)
            sc = ax.scatter(x[valid], y[valid], c=lk[valid], cmap=cmap, vmin=0, vmax=1,
                            s=6, alpha=0.7, linewidths=0)
            # small colorbar to the right
            cb = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.02)
            cb.set_label("likelihood", fontsize=7)
            cb.ax.tick_params(labelsize=7)
        else:
            ax.scatter(x[valid], y[valid], color="#4c72b0", s=6, alpha=0.7)
        ax.set_xlabel("x", fontsize=8)
        ax.set_ylabel("y", fontsize=8)
        ax.set_title(kp.replace("_", " "), fontsize=9, loc="left")
        ax.grid(True, ls="--", alpha=0.4)
    # annotate NA fraction
    ax.text(0.98, 0.92, f"NA {na_frac:.1%}", transform=ax.transAxes,
            ha="right", va="top", fontsize=7, color="gray",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.6, linewidth=0))

# remove unused axes
for j in range(nplots, nrows * ncols):
    fig.delaxes(axes[j])

fig.suptitle("Pose dataframe — per-column histograms + keypoint scatters", fontsize=14)
plt.show()
pose.iloc[1000]

In [ ]:
frames = session_modality_from_nas(session_dirs[0], 'facecam_frames', )
frames = frames[:500]


In [ ]:
# print(session_names)
# plt.hist(np.diff(behavior[angle_col]), bins=300)

print(frames.shape)
print(pose.shape)
print(pose.columns)

# Create animation with frames as background
ani = animate_body_points(pose.reset_index(), frames=frames, start_idx=100, end_idx=500, interval=1000/24)

from IPython.display import HTML
HTML(ani.to_jshtml())

In [ ]:
pose.columns

In [ ]:
for column in pose.columns:
    if column.startswith("facecam") and "_x" not in column and "_y" not in column:
        plt.figure()
        plt.hist(pose[column].dropna(), bins=1000)
        plt.title(f"Histogram of {column}")
        plt.show()
for point_name in ["facecam_pose_nose", "facecam_pose_neck", "facecam_pose_body_1"]:
    x_col = f"{point_name}_x"
    y_col = f"{point_name}_y"
    plt.figure()
    print(pose[x_col])
    print(pose[y_col].dropna())
    print(pose[x_col])
    print(pose[y_col].dropna())
    plt.scatter(pose[x_col].dropna(), pose[y_col].dropna(), s=1, alpha=.1)
    plt.title(f"Scatter plot of {point_name} positions")
    plt.xlabel("X position")
    plt.ylabel("Y position")
    plt.show()
    break

# iter _likelihood columns and plot histograms for each in different color
plt.figure()
for column in pose.columns:
    if column.startswith("facecam") and column.endswith("_likelihood"):
        plt.hist(pose[column].dropna(), bins=100, alpha=0.7, label=column)
        plt.title(f"Histogram of {column}")
        plt.xlabel("Likelihood")
        plt.ylabel("Frequency")
plt.legend()
plt.show()
        

In [ ]:
# fr_raw = analytics.get_analytics('FiringRate40msHz', session_names=session_names)

behavior = analytics.get_analytics('BehaviorFramewise', session_names=['2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min'], columns=[
    "frame_ephys_timestamp",
    # action based
    "frame_velocity",
    "frame_acceleration",
    "frame_raw",
    "frame_yaw",
    "frame_pitch",
    "lick_count",
    
    # state based
    "frame_position",
    # "reward-removed_count",
    "reward-sound_count",
    "reward-valve-open_count",
    
    
    
    "facecam_pose_nose_x",
    "facecam_pose_nose_y",
    # facecam_pose_nose_likelihood
    "facecam_pose_neck_x",
    "facecam_pose_neck_y",
    # facecam_pose_neck_likelihood
    "facecam_pose_body_1_x",
    "facecam_pose_body_1_y",
    # facecam_pose_body_1_likelihood
    # facecam_pose_body_2_x
    # facecam_pose_body_2_y
    # facecam_pose_body_2_likelihood
    "facecam_pose_body_3_x",
    "facecam_pose_body_3_y",
    # facecam_pose_body_3_likelihood
    "facecam_pose_nose_neck_length",
    "facecam_pose_neck_body1_length",
    "facecam_pose_nose_neck_body1_angle",
    "facecam_pose_body1_body2_body3_angle",

])
behavior

In [ ]:
for column in behavior.columns:
    if column.startswith("facecam") and "_x" not in column and "_y" not in column:
        plt.figure()
        plt.hist(behavior[column].dropna(), bins=1000)
        plt.title(f"Histogram of {column}")
        plt.show()
for point_name in ["facecam_pose_nose", "facecam_pose_neck", "facecam_pose_body_1"]:
    x_col = f"{point_name}_x"
    y_col = f"{point_name}_y"
    plt.figure()
    print(behavior[x_col])
    # print(behavior[y_col].dropna())
    print(behavior[x_col])
    # print(behavior[y_col].dropna())
    plt.scatter(behavior[x_col].dropna(), behavior[y_col].dropna(), s=1, alpha=.1)
    plt.title(f"Scatter plot of {point_name} positions")
    plt.xlabel("X position")
    plt.ylabel("Y position")
    plt.show()
    break

In [ ]:
framew_beh = analytics.get_analytics('BehaviorFramewise', animal_ids=[6], paradigm_ids=[1100])
# framew_beh.index = framew_beh.droplevel(['animal_id', 'entry_id'])
# framew_beh

In [ ]:
s_id = 0
framew_beh.columns

In [ ]:
plt.close('all')
for s_id in framew_beh.index.unique('session_id'):
    if not (s_id >20):
        continue
    session_data = framew_beh[framew_beh.index.get_level_values('session_id') == s_id]
    
    frame_lengths = np.diff(session_data.frame_pc_timestamp)
    ratio = (frame_lengths>30_000).sum() / frame_lengths.size
    frame_lengths = np.clip(frame_lengths, 0, 45_000)
    plt.hist(frame_lengths  , bins=300, label=f'Session {s_id}, Ratio {ratio:.2f}', histtype='step', alpha=0.7)
    plt.yscale("log")
plt.legend(fontsize='small', ncol=2)
plt.show()

In [ ]:
plt.plot(session_beh.frame_position.iloc[:2000].values)

In [ ]:
plt.close('all')
session_beh = framew_beh.xs(s_id, level='session_id')
for trial in session_beh['trial_id'].unique():
    if trial == -1:
        continue
    trial_d = session_beh[session_beh['trial_id'] == trial]
    print(trial_d.shape, trial)
    t_start = trial_d['frame_pc_timestamp'].iloc[0]
    # t_end = trial_d['frame_pc_timestamp'].iloc[-1]
    positions = trial_d[['frame_position']]
    velocities = trial_d[['frame_velocity']]
    time = trial_d['frame_pc_timestamp'] - t_start
    # print(positions)
    # print(time)
    
    # draw line
    # plt.plot(time, positions, label=f'Trial {trial}')
    plt.plot(time, velocities, label=f'Trial {trial}')
    if trial >=12:
        break
# plt.xlim(0, 20_000_000)
plt.ylim(0, 100)
plt.show()